In [1]:
import json
import re
import time
from pathlib import Path
from typing import Dict, List, Any, Optional

import pandas as pd

try:
    import yaml
except ImportError:
    yaml = None

try:
    import mammoth
except ImportError:
    mammoth = None


def extract_and_setup_constitutional_questions(
    docx_path: str = "Questions.docx",
    config_path: str = "question_chunks.yaml"
) -> Optional[str]:
    """
    Full pipeline for extracting constitutional questions from a DOCX and organizing them.
    Includes a timer for tracking execution time.

    Args:
        docx_path: Path to the DOCX file containing questions
        config_path: Path to the chunks config (YAML or JSON)

    Returns:
        Path to the output JSON file if successful, otherwise None
    """
    start_time = time.perf_counter()

    print(f"Reading DOCX: {docx_path}")
    try:
        if mammoth is None:
            raise ImportError("mammoth library required. Install with: pip install mammoth")
        
        with open(docx_path, "rb") as docx_file:
            result = mammoth.extract_raw_text(docx_file)
            docx_text = result.value
        
        print(f"Extracted {len(docx_text)} characters from DOCX")
    except FileNotFoundError:
        print(f"Error: Could not find '{docx_path}'. Please check the file path.")
        return None
    except Exception as e:
        print(f"Error reading DOCX: {e}")
        return None

    organizer = ConstitutionalQuestionOrganizer(config_path=config_path)

    print("Extracting questions from DOCX text...")
    all_questions = organizer.extract_all_questions_from_docx(docx_text)
    print(f"Found {len(all_questions)} questions")

    print("Organizing questions into thematic chunks...")
    organized = organizer.organize_into_chunks(all_questions)

    # Generate versioned filenames to prevent overwriting
    output_file = organizer._get_versioned_filename("organized_constitutional_questions.json")
    organizer.save_organized_questions(organized, output_file)

    # Export question IDs to text file
    id_output_file = organizer._get_versioned_filename("question_ids_organized.txt")
    organizer.export_question_ids_to_txt(organized, all_questions, id_output_file)

    elapsed_time = time.perf_counter() - start_time

    print("\n" + "=" * 60)
    print("EXTRACTION COMPLETE")
    print("=" * 60)

    total_questions = sum(len(chunk["questions"]) for chunk in organized.values())
    print(f"Total questions extracted: {total_questions}")
    print(f"Organized into {len(organized)} chunks")
    print(f"JSON saved to: {output_file}")
    print(f"Question IDs exported to: {id_output_file}")
    print(f"Total extraction time: {elapsed_time:.2f} seconds")

    print("\nChunk breakdown:")
    for chunk_id, chunk_data in organized.items():
        print(f"  {chunk_id:<25} {len(chunk_data['questions']):>3} questions - {chunk_data['title']}")

    return output_file


class ConstitutionalQuestionOrganizer:
    """
    Organizer for extracting and categorizing constitutional questions from text.
    """

    def __init__(self, config_path: str = "question_chunks.yaml") -> None:
        self.question_chunks = self._load_chunks_config(config_path)

    # ------------------------
    # File Versioning
    # ------------------------

    def _get_versioned_filename(self, base_filename: str) -> str:
        """Generate a versioned filename to prevent overwriting existing files."""
        path = Path(base_filename)
        stem = path.stem
        suffix = path.suffix
        
        if not path.exists():
            return base_filename
        
        version = 1
        while True:
            versioned_name = f"{stem}_v{version}{suffix}"
            if not Path(versioned_name).exists():
                return versioned_name
            version += 1

    # ------------------------
    # Config Loader
    # ------------------------

    def _load_chunks_config(self, config_path: str) -> dict:
        """Load thematic chunks from YAML or JSON config."""
        config_file = Path(config_path)
        if not config_file.exists():
            raise FileNotFoundError(f"Config file not found: {config_path}")

        if config_file.suffix.lower() in [".yaml", ".yml"]:
            if not yaml:
                raise ImportError("pyyaml is required to read YAML configs. Install with `pip install pyyaml`.")
            with open(config_file, "r", encoding="utf-8") as f:
                return yaml.safe_load(f)

        elif config_file.suffix.lower() == ".json":
            with open(config_file, "r", encoding="utf-8") as f:
                return json.load(f)

        else:
            raise ValueError(f"Unsupported config format: {config_file.suffix}. Use .yaml, .yml, or .json")

    # ------------------------
    # Question Extraction (DOCX)
    # ------------------------
    
    def _parse_conditional_text(self, cond_text: Optional[str]) -> Dict[str, Any]:
        """
        [MOVED FROM extract_dependencies.py]
        Parses the raw conditional string into a structured dictionary.
        """
        if not cond_text:
            return {"raw": None, "depends_on": [], "condition_expression": None}

        raw = cond_text.strip()
        # remove leading phrase
        text = re.sub(r"\(Asked only if\s*", "", raw, flags=re.IGNORECASE)
        text = text.strip(" )")

        # extract variable references (v## or UPPER_CODES)
        depends_on = re.findall(r"\b(v\d+|[A-Z_]{2,})\b", text)

        # normalize to python-like expression
        expr = re.sub(r"(?i)is answered", "==", text)
        expr = re.sub(r"(?i)answered", "==", expr)
        expr = re.sub(r"(?i)or if", "or", expr)
        expr = re.sub(r"(?i)and if", "and", expr)
        expr = expr.replace(",", " or ")
        expr = re.sub(r"\s+", " ", expr).strip()
        expr = re.sub(r"\b(or|and)\s+\1\b", r"\1", expr)
        expr = re.sub(r"\s*(or|and|,|\.)\s*$", "", expr)
        expr = expr.replace("= =", "==").strip()

        return {
            "raw": raw,
            "depends_on": sorted(set(depends_on)),
            "condition_expression": expr
        }

    def extract_all_questions_from_docx(self, docx_text: str) -> Dict[str, Dict[str, Any]]:
        """Parse questions and answer options from extracted DOCX text."""
        all_questions = {}
        lines = docx_text.split("\n")

        current_question = None
        in_instructions = False
        i = 0
        order_index = 0 

        while i < len(lines):
            line = lines[i].strip()
            
            if not line:
                i += 1
                continue

            question_match = self._match_question_line_docx(line)
            if question_match:
                # Before moving to the next question, check if current one needs a code fix
                if current_question:
                    self._apply_code_fallback(current_question)
                    all_questions[current_question["id"]] = current_question

                v_num, code, question_text = self._parse_question_match_docx(question_match)
                
                j = i + 1
                full_question = question_text
                while j < len(lines):
                    next_line = lines[j].strip()
                    if not next_line or self._match_option_line_docx(next_line) or \
                       self._match_question_line_docx(next_line) or next_line.lower().startswith("instructions:"):
                        break
                    full_question += " " + next_line
                    j += 1

                raw_cond_text = self._extract_conditional_from_text(full_question)
                parsed_conditional = self._parse_conditional_text(raw_cond_text)
                cleaned_question_text = self._clean_question_text(full_question)

                current_question = {
                    "id": f"v{v_num}",
                    "code": code if code != "UNKNOWN" else None,
                    "question": cleaned_question_text,
                    "options": [],
                    "conditional": parsed_conditional,
                    "instructions": None,
                    "multi_select": "check all that apply" in full_question.lower(),
                    "category": None,
                    "order_index": order_index 
                }
                order_index += 1
                in_instructions = False
                i = j - 1

            elif current_question and self._match_option_line_docx(line):
                in_instructions = False
                option_match = self._match_option_line_docx(line)
                option_num, option_content = option_match.groups()
                
                option_code = self._extract_option_code(option_content)
                
                # Enhanced cleaning: Remove the [CODE] and any surrounding punctuation/whitespace
                cleaned_text = option_content.replace('\u00ad', '')
                if option_code:
                    # Removes "[CODE]", " - [CODE]", " [CODE]" etc.
                    cleaned_text = re.sub(r'\s*[–-]?\s*\[%s\]' % re.escape(option_code), '', cleaned_text)
                
                cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

                current_question["options"].append({
                    "number": option_num,
                    "text": cleaned_text,
                    "code": option_code,
                })

            elif current_question and line.lower().startswith("instructions:"):
                in_instructions = True
                instruction_text = line[len("Instructions:"):].strip()
                if current_question["instructions"]:
                    current_question["instructions"] += " " + instruction_text
                else:
                    current_question["instructions"] = instruction_text

            elif current_question and in_instructions:
                current_question["instructions"] += " " + line

            i += 1

        # Final question wrap-up
        if current_question:
            self._apply_code_fallback(current_question)
            all_questions[current_question["id"]] = current_question

        return all_questions

    def _apply_code_fallback(self, question: Dict[str, Any]) -> None:
        """
        If the question has no code, attempt to derive one from the option codes.
        E.g., if options have [CHALLEG_1, CHALLEG_2], question code becomes CHALLEG.
        """
        if question.get("code") is None and question.get("options"):
            # Collect all valid option codes that look like variables (contain letters)
            opt_codes = [o["code"] for o in question["options"] if o["code"] and re.search(r'[A-Z]', o["code"])]
            
            if opt_codes:
                # Find common prefix by stripping trailing underscores and numbers
                # e.g., "CHALLEG_1" -> "CHALLEG"
                stems = [re.sub(r'(_\d+|\d+)$', '', c) for c in opt_codes]
                # Use the most common stem found (usually they are all the same)
                common_stem = max(set(stems), key=stems.count)
                question["code"] = common_stem
                question["is_derived_code"] = True # Flag for traceability
    
    # ------------------------
    # Organization
    # ------------------------

    def organize_into_chunks(self, all_questions: Dict[str, Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
        """Group questions into thematic chunks using codes and keywords."""
        organized, unassigned = {}, []

        for chunk_id, chunk_info in self.question_chunks.items():
            organized[chunk_id] = {
                "title": chunk_info["title"],
                "description": chunk_info["description"],
                "questions": [],
            }

        for question_id, question in all_questions.items():
            assigned = False
            q_code, q_text = question.get("code", "").upper() if question.get("code") else "", question.get("question", "").lower()

            for chunk_id, chunk_info in self.question_chunks.items():
                if any(code in q_code for code in chunk_info["codes"]) or \
                   any(keyword in q_text for keyword in chunk_info.get("keywords", [])):
                    organized[chunk_id]["questions"].append(question)
                    question["category"] = chunk_id
                    assigned = True
                    break

            if not assigned:
                unassigned.append(question)

        if unassigned:
            organized["miscellaneous"] = {
                "title": "Miscellaneous Questions",
                "description": "Questions that could not be categorized",
                "questions": unassigned,
            }

        return organized

    def save_organized_questions(self, organized_questions: Dict[str, Any], filepath: str) -> None:
        """Save organized questions and metadata as JSON."""
        summary = {
            "total_chunks": len(organized_questions),
            "total_questions": sum(len(chunk["questions"]) for chunk in organized_questions.values()),
            "chunk_summary": {
                chunk_id: {
                    "title": chunk_data["title"],
                    "question_count": len(chunk_data["questions"]),
                    "sample_codes": [q["code"] for q in chunk_data["questions"][:3]],
                }
                for chunk_id, chunk_data in organized_questions.items()
            },
        }

        output = {"metadata": summary, "chunks": organized_questions}

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2, ensure_ascii=False)

    def export_question_ids_to_txt(
        self, 
        organized_questions: Dict[str, Any], 
        all_questions: Dict[str, Dict[str, Any]],
        filepath: str
    ) -> None:
        """
        Export all question IDs to a text file, organized by YAML categories.
        Also includes a section showing conditional dependencies between questions.
        """
        with open(filepath, "w", encoding="utf-8") as f:
            f.write("=" * 70 + "\n")
            f.write("CONSTITUTIONAL QUESTIONS - ORGANIZED QUESTION IDs\n")
            f.write("=" * 70 + "\n\n")

            total_count = 0

            # Section 1: Questions organized by category
            f.write("QUESTIONS BY CATEGORY:\n")
            f.write("-" * 70 + "\n\n")

            for chunk_id, chunk_data in organized_questions.items():
                questions = chunk_data["questions"]
                total_count += len(questions)

                f.write(f"[{chunk_id.upper()}] {chunk_data['title']}\n")
                f.write(f"Count: {len(questions)} questions\n\n")
                
                for q in questions:
                    q_id = q["id"]
                    q_code = q.get("code", "UNKNOWN")
                    f.write(f"  {q_id} - [{q_code}]\n")
                
                f.write("\n")

            # Section 2: Complete alphabetical list with codes
            f.write("\n" + "=" * 70 + "\n")
            f.write("COMPLETE ALPHABETICAL LIST:\n")
            f.write("=" * 70 + "\n\n")

            all_ids = sorted(all_questions.keys(), key=lambda x: int(x[1:]))
            f.write(f"Total Questions: {len(all_ids)}\n\n")
            
            for q_id in all_ids:
                q = all_questions[q_id]
                q_code = q.get("code", "UNKNOWN")
                f.write(f"{q_id} - [{q_code}]\n")

            # Section 3: Conditional dependencies
            f.write("\n\n" + "=" * 70 + "\n")
            f.write("CONDITIONAL DEPENDENCIES:\n")
            f.write("=" * 70 + "\n")
            f.write("(Questions that are only asked based on answers to other questions)\n\n")

            # --- UPDATED: Use new helper ---
            conditional_map = self._extract_conditional_dependencies(all_questions)

            if conditional_map:
                for question_id, conditional_info in sorted(conditional_map.items(), key=lambda x: int(x[0][1:])):
                    q = all_questions[question_id]
                    q_code = q.get("code", "UNKNOWN")
                    
                    f.write(f"{question_id} - [{q_code}]:\n")
                    f.write(f"  Condition: {conditional_info['condition']}\n")
                    if conditional_info['depends_on']:
                        depends_str = ", ".join([
                            f"{dep_id} - [{all_questions.get(dep_id, {}).get('code', 'UNKNOWN')}]" 
                            if dep_id.startswith('v') 
                            else dep_id
                            for dep_id in conditional_info['depends_on']
                        ])
                        f.write(f"  Depends on: {depends_str}\n")
                    f.write("\n")
            else:
                f.write("No conditional dependencies found.\n")

            # Summary footer
            f.write("\n" + "=" * 70 + "\n")
            f.write(f"SUMMARY: {len(all_ids)} total questions across {len(organized_questions)} categories\n")
            f.write(f"Conditional questions: {len(conditional_map)}\n")
            f.write("=" * 70 + "\n")

    def _extract_conditional_dependencies(self, all_questions: Dict[str, Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
        """
        Extract conditional dependencies between questions
        [UPDATED] to read from the pre-parsed conditional dictionary.
        """
        conditional_map = {}

        for question_id, question in all_questions.items():
            # The 'conditional' field is now a dictionary
            conditional_dict = question.get("conditional", {})
            
            # Check if there's a 'raw' text, meaning it *is* conditional
            if conditional_dict.get("raw"):
                conditional_map[question_id] = {
                    "condition": conditional_dict["raw"],
                    "depends_on": conditional_dict.get("depends_on", [])
                }

        return conditional_map

    # ------------------------
    # Regex Helpers (DOCX)
    # ------------------------

    def _match_question_line_docx(self, line: str) -> Optional[re.Match]:
        """Match question lines in DOCX format."""
        # Pattern: v70. [AMEND] -- Does the...
        # Pattern: v74. Who is allowed to...
        patterns = [
            r"^v(\d+)\.\s*\[([A-Z_]+)\]\s*[–-]+\s*(.+)",  # With code
            r"^v(\d+)\.\s+(.+)",  # Without code, simplified to be more general
        ]
        for pattern in patterns:
            match = re.match(pattern, line, re.IGNORECASE)
            if match:
                return match
        return None

    def _match_option_line_docx(self, line: str) -> Optional[re.Match]:
        """Match option lines."""
        # Matches lines like "1. Yes" or "96. other..."
        pattern = r"^(\d+)\.\s*(.+)"
        return re.match(pattern, line)


    def _parse_question_match_docx(self, match: re.Match) -> tuple:
        """Parse question match into components."""
        groups = match.groups()
        if len(groups) == 3:
            v_num, code, question_text = groups
        else:
            v_num, question_text = groups
            code = "UNKNOWN"
        return v_num, code.strip(), question_text.strip()

    def _extract_conditional_from_text(self, text: str) -> Optional[str]:
        """Extract conditional clause from question text."""
        match = re.search(r"\(Asked only if[^)]+\)", text, re.IGNORECASE)
        return match.group(0) if match else None

    def _clean_question_text(self, text: str) -> str:
        """Clean question text by removing conditional clauses and normalizing whitespace."""
        text = re.sub(r"\(Asked only if[^)]+\)", "", text, flags=re.IGNORECASE)
        
        # Remove soft hyphens (a common DOCX artifact)
        text = text.replace('\u00ad', '')
        # Normalize all whitespace (including tabs, newlines etc.) to a single space
        text = re.sub(r"\s+", " ", text).strip()
        
        return text

    def _extract_option_code(self, option_text: str) -> Optional[str]:
        """Extract code from option text (e.g., [AMNDPROP_1])."""
        match = re.search(r"\[([A-Z_0-9]+)\]", option_text)
        return match.group(1) if match else None


In [2]:
# Usage
if __name__ == "__main__":
    extract_and_setup_constitutional_questions(
        docx_path="Questions_roots_added.docx",
        config_path="question_chunks.yaml"
    )

Reading DOCX: Questions_roots_added.docx
Extracted 85076 characters from DOCX
Extracting questions from DOCX text...
Found 147 questions
Organizing questions into thematic chunks...

EXTRACTION COMPLETE
Total questions extracted: 147
Organized into 9 chunks
JSON saved to: organized_constitutional_questions.json
Question IDs exported to: question_ids_organized.txt
Total extraction time: 0.59 seconds

Chunk breakdown:
  amendment_process           6 questions - Constitutional Amendment Process
  executive_branch           41 questions - Executive Branch Structure and Powers
  legislative_branch         25 questions - Legislative Branch Structure and Powers
  judicial_branch            33 questions - Judicial Branch and Court System
  elections_voting_parties   12 questions - Elections, Voting, and Political Participation
  fundamental_rights         17 questions - Fundamental Rights and Civil Liberties
  government_institutions     8 questions - Government Structure and Institutions
  mi

The below script analyzes the generated questions JSON file to find and report any "broken" conditional dependencies where a question refers to a root question ID or code that doesn't actually exist in the file.

In [6]:

import json
import re
from pathlib import Path
from typing import Dict, Any, Set, List

def find_missing_conditional_roots(json_filepath: str) -> None:
    """
    [UPDATED]
    Reads an organized questions JSON file and identifies conditional questions
    whose root dependencies (e.g., 'AMNDAPPR' or 'v123') are not
    found in the list of extracted questions.
    
    This now reads the pre-parsed 'depends_on' list from the JSON.
    """
    print(f"Analyzing '{json_filepath}' for missing conditional roots...")

    # --- Step 1: Load the JSON file ---
    json_path = Path(json_filepath)
    if not json_path.exists():
        print(f"Error: File not found at {json_filepath}")
        return

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        if "chunks" not in data:
            print("Error: JSON format is incorrect. Expected a top-level 'chunks' key.")
            return
            
    except json.JSONDecodeError as e:
        print(f"Error reading JSON file: {e}")
        return
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return

    # --- Step 2: Build sets of all known IDs and codes ---
    all_questions: Dict[str, Dict[str, Any]] = {}
    known_v_ids: Set[str] = set()
    known_codes: Set[str] = set()

    for chunk_data in data.get("chunks", {}).values():
        for q in chunk_data.get("questions", []):
            if "id" in q:
                all_questions[q["id"]] = q
                known_v_ids.add(q["id"])
                if q.get("code"):
                    known_codes.add(q["code"])

    if not all_questions:
        print("No questions found in the JSON file.")
        return

    # Combine all known identifiers
    all_known_identifiers = known_v_ids.union(known_codes)
    print(f"Loaded {len(all_questions)} questions. Checking dependencies against {len(all_known_identifiers)} known identifiers...")

    # --- Step 3: Find and check conditional questions ---
    missing_roots_report: Dict[str, Dict[str, Any]] = {}


    for q_id, q_data in all_questions.items():
        # [UPDATED] Read from the conditional dictionary
        conditional_dict = q_data.get("conditional", {})
        
        if conditional_dict.get("raw"):
            # [UPDATED] Get the pre-parsed list of dependencies
            mentioned_dependencies = set(conditional_dict.get("depends_on", []))
            
            if not mentioned_dependencies:
                continue
            
            # --- Step 4: Compare mentioned vs. known sets ---
            # [UPDATED] Simpler check
            all_missing = list(mentioned_dependencies - all_known_identifiers)
            
            if all_missing:
                missing_roots_report[q_id] = {
                    "code": q_data.get("code", "N/A"),
                    "condition": conditional_dict.get("raw"),
                    "missing": all_missing
                }

    # --- Step 5: Report the findings ---
    if not missing_roots_report:
        print("\n" + "="*60)
        print("ANALYSIS COMPLETE: All conditional roots were found.")
        print("="*60)
    else:
        print("\n" + "="*60)
        print(f"ANALYSIS COMPLETE: Found {len(missing_roots_report)} questions with missing roots.")
        print("="*60)
        
        for q_id, info in missing_roots_report.items():
            print(f"\n  [!] Question: {q_id} [{info['code']}]")
            print(f"      Condition: {info['condition']}")
            print(f"      Missing Root(s): {', '.join(info['missing'])}")


if __name__ == "__main__":
    # --------------------------------------------------------------------
    # --- RUN THIS SCRIPT ---
    # --------------------------------------------------------------------
    #
    # 1. Make sure you have run the *other* script first to generate
    #    the JSON file.
    #
    # 2. Change the 'json_to_analyze' variable below to match the
    #    exact filename of your generated JSON output.
    #
    # --------------------------------------------------------------------

    # *** UPDATE THIS FILENAME ***
    # This should be the path to the JSON file created by the
    # 'extract_and_setup_constitutional_questions' function.
    json_to_analyze = "organized_constitutional_questions.json"

    # Example: If your file was versioned, it might be:
    # json_to_analyze = "organized_constitutional_questions_v1.json"

    find_missing_conditional_roots(json_to_analyze)

Analyzing 'organized_constitutional_questions.json' for missing conditional roots...
Loaded 147 questions. Checking dependencies against 257 known identifiers...

ANALYSIS COMPLETE: All conditional roots were found.
